In [5]:
# ==========================================
# Profile-Aware Glasses Detection (Image)
# ==========================================

import cv2
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import models
from ultralytics import YOLO
import mediapipe as mp
from PIL import Image

# ==========================================
# Config
# ==========================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IMAGE_PATH = r"C:\Users\Acer\Desktop\22222.jpg"
GLASSES_MODEL_PATH = r"D:\VISION_2\Final_Glass_Detection\resnet18_glasses.pth"

# Thresholds
FRONTAL_THRESH = 0.65
PROFILE_THRESH = 0.50

# ==========================================
# Load Models
# ==========================================

# YOLO person detector
yolo = YOLO("yolov8n.pt")

# MediaPipe face detector
mp_face = mp.solutions.face_detection
face_detector = mp_face.FaceDetection(
    model_selection=1,
    min_detection_confidence=0.6
)

# Glasses classifier (ResNet18)
glasses_model = models.resnet18(weights=None)
glasses_model.fc = nn.Linear(glasses_model.fc.in_features, 2)
glasses_model.load_state_dict(torch.load(GLASSES_MODEL_PATH, map_location=DEVICE))
glasses_model = glasses_model.to(DEVICE)
glasses_model.eval()

# ==========================================
# Transform
# ==========================================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ==========================================
# Quality Gate (فقط junk)
# ==========================================
def quality_gate(face_crop):
    h, w = face_crop.shape[:2]

    if h < 48 or w < 48:
        return False

    gray = cv2.cvtColor(face_crop, cv2.COLOR_BGR2GRAY)
    if gray.mean() < 45:
        return False

    return True

# ==========================================
# Glasses Prediction
# ==========================================
def predict_glasses(face_crop):
    img = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(img)
    img = transform(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = glasses_model(img)
        probs = torch.softmax(outputs, dim=1)
        conf, pred = torch.max(probs, 1)

    label = "glasses" if pred.item() == 0 else "no_glasses"
    return label, conf.item()

# ==========================================
# Load Image
# ==========================================
frame = cv2.imread(IMAGE_PATH)
H, W = frame.shape[:2]

# UI scale
UI_SCALE = max(0.35, min(W / 800, 0.7))
UI_THICK = 1 if W < 600 else 2
LINE_GAP = int(22 * UI_SCALE)

# Counters
count_glasses = 0
count_noglasses = 0
count_unknown = 0

# ==========================================
# Run Pipeline
# ==========================================
results = yolo(frame)

for box in results[0].boxes:
    cls = int(box.cls[0])
    if cls != 0:
        continue

    x1, y1, x2, y2 = map(int, box.xyxy[0])
    person_crop = frame[y1:y2, x1:x2]

    rgb_person = cv2.cvtColor(person_crop, cv2.COLOR_BGR2RGB)
    faces = face_detector.process(rgb_person)

    if not faces.detections:
        continue

    for det in faces.detections:
        bbox = det.location_data.relative_bounding_box
        ph, pw, _ = person_crop.shape

        fx1 = int(bbox.xmin * pw)
        fy1 = int(bbox.ymin * ph)
        fx2 = int((bbox.xmin + bbox.width) * pw)
        fy2 = int((bbox.ymin + bbox.height) * ph)

        # clamp
        fx1 = max(0, fx1)
        fy1 = max(0, fy1)
        fx2 = min(pw, fx2)
        fy2 = min(ph, fy2)

        face_crop = person_crop[fy1:fy2, fx1:fx2]
        if face_crop.size == 0:
            continue

        conf = None

        # ========== Gate ==========
        if not quality_gate(face_crop):
            label = "unknown"
        else:
            label, conf = predict_glasses(face_crop)

            # Profile-aware threshold
            aspect = (fx2 - fx1) / (fy2 - fy1 + 1e-6)
            thresh = PROFILE_THRESH if aspect < 0.75 else FRONTAL_THRESH

            if conf < thresh:
                label = "unknown"

        # ========== Color & Count ==========
        if label == "glasses":
            color = (0, 255, 0)
            count_glasses += 1
        elif label == "no_glasses":
            color = (255, 0, 0)
            count_noglasses += 1
        else:
            color = (0, 0, 255)
            count_unknown += 1

        # ========== Draw ==========
        text = f"{label} ({conf:.2f})" if conf is not None and label != "unknown" else label

        cv2.rectangle(
            frame,
            (x1 + fx1, y1 + fy1),
            (x1 + fx2, y1 + fy2),
            color,
            2
        )

        cv2.putText(
            frame,
            text,
            (x1 + fx1, max(15, y1 + fy1 - 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            UI_SCALE,
            color,
            UI_THICK
        )

        break  

# ==========================================
# HUD
# ==========================================
overlay = frame.copy()
cv2.rectangle(overlay, (5, 5), (220, int(30 + 4 * LINE_GAP)), (0, 0, 0), -1)
frame = cv2.addWeighted(overlay, 0.4, frame, 0.6, 0)

cv2.putText(frame, f"Total: {count_glasses + count_noglasses + count_unknown}",
            (10, int(30 * UI_SCALE)),
            cv2.FONT_HERSHEY_SIMPLEX, UI_SCALE, (255,255,255), UI_THICK)

cv2.putText(frame, f"Glasses: {count_glasses}",
            (10, int(30 * UI_SCALE + LINE_GAP)),
            cv2.FONT_HERSHEY_SIMPLEX, UI_SCALE, (0,255,0), UI_THICK)

cv2.putText(frame, f"No Glasses: {count_noglasses}",
            (10, int(30 * UI_SCALE + 2 * LINE_GAP)),
            cv2.FONT_HERSHEY_SIMPLEX, UI_SCALE, (255,0,0), UI_THICK)

cv2.putText(frame, f"Unknown: {count_unknown}",
            (10, int(30 * UI_SCALE + 3 * LINE_GAP)),
            cv2.FONT_HERSHEY_SIMPLEX, UI_SCALE, (0,0,255), UI_THICK)

# ==========================================
# Show
# ==========================================
cv2.imshow("Profile-Aware Glasses Detection", frame)
cv2.waitKey(0)
cv2.destroyAllWindows()



0: 384x640 2 persons, 1 refrigerator, 103.4ms
Speed: 43.8ms preprocess, 103.4ms inference, 2.6ms postprocess per image at shape (1, 3, 384, 640)


In [7]:
# ==========================================
# Profile-Aware Glasses Detection (Video File)
# ==========================================

import cv2
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import models
from ultralytics import YOLO
import mediapipe as mp
from PIL import Image
import time
import os

# =====================
# Config
# =====================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GLASSES_MODEL_PATH = r"C:\Users\Acer\Desktop\new_try\resnet18_glasses.pth"

VIDEO_PATH = r"C:\Users\Acer\Downloads\Video\Spectacles Girl 4K Stock Video - Download Video Clip Now - Eyeglasses, Getting Dressed, Applying - iStock.mp4"
OUTPUT_PATH = r"C:\Users\Acer\Desktop\output_profile_aware.mp4"  # optional
SAVE_OUTPUT = True

YOLO_MODEL = "yolov8n.pt"

FRONTAL_THRESH = 0.65
PROFILE_THRESH = 0.50
ASPECT_PROFILE_CUTOFF = 0.75

# =====================
# Load Models
# =====================
yolo = YOLO(YOLO_MODEL)

mp_face = mp.solutions.face_detection
face_detector = mp_face.FaceDetection(model_selection=1, min_detection_confidence=0.6)

glasses_model = models.resnet18(weights=None)
glasses_model.fc = nn.Linear(glasses_model.fc.in_features, 2)
glasses_model.load_state_dict(torch.load(GLASSES_MODEL_PATH, map_location=DEVICE))
glasses_model = glasses_model.to(DEVICE)
glasses_model.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

def quality_gate(face_crop):
    h, w = face_crop.shape[:2]
    if h < 48 or w < 48:
        return False
    gray = cv2.cvtColor(face_crop, cv2.COLOR_BGR2GRAY)
    if gray.mean() < 45:
        return False
    return True

def predict_glasses(face_crop):
    img = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(img)
    img = transform(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        outputs = glasses_model(img)
        probs = torch.softmax(outputs, dim=1)
        conf, pred = torch.max(probs, 1)
    label = "glasses" if pred.item() == 0 else "no_glasses"
    return label, float(conf.item())

# =====================
# Video IO
# =====================
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError("Cannot open video")

W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps_in = cap.get(cv2.CAP_PROP_FPS)
fps_in = fps_in if fps_in and fps_in > 0 else 25

writer = None
if SAVE_OUTPUT:
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps_in, (W, H))

cv2.namedWindow("Video Profile-Aware Glasses", cv2.WINDOW_NORMAL)

last_time = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    UI_SCALE = max(0.35, min(W / 800, 0.7))
    UI_THICK = 1 if W < 600 else 2
    LINE_GAP = int(22 * UI_SCALE)

    count_glasses = 0
    count_noglasses = 0
    count_unknown = 0

    results = yolo(frame)

    for box in results[0].boxes:
        cls = int(box.cls[0])
        if cls != 0:
            continue

        x1, y1, x2, y2 = map(int, box.xyxy[0])
        person_crop = frame[y1:y2, x1:x2]
        if person_crop.size == 0:
            continue

        rgb_person = cv2.cvtColor(person_crop, cv2.COLOR_BGR2RGB)
        faces = face_detector.process(rgb_person)

        if not faces.detections:
            continue

        det = faces.detections[0]
        bbox = det.location_data.relative_bounding_box
        ph, pw, _ = person_crop.shape

        fx1 = int(bbox.xmin * pw)
        fy1 = int(bbox.ymin * ph)
        fx2 = int((bbox.xmin + bbox.width) * pw)
        fy2 = int((bbox.ymin + bbox.height) * ph)

        pad = 8
        fx1 = max(0, fx1 - pad)
        fy1 = max(0, fy1 - pad)
        fx2 = min(pw, fx2 + pad)
        fy2 = min(ph, fy2 + pad)

        face_crop = person_crop[fy1:fy2, fx1:fx2]
        if face_crop.size == 0:
            continue

        conf = None
        label = "unknown"

        if quality_gate(face_crop):
            pred_label, pred_conf = predict_glasses(face_crop)
            conf = pred_conf

            aspect = (fx2 - fx1) / (fy2 - fy1 + 1e-6)
            thresh = PROFILE_THRESH if aspect < ASPECT_PROFILE_CUTOFF else FRONTAL_THRESH

            if pred_conf >= thresh:
                label = pred_label
            else:
                label = "unknown"

        if label == "glasses":
            color = (0, 255, 0); count_glasses += 1
        elif label == "no_glasses":
            color = (255, 0, 0); count_noglasses += 1
        else:
            color = (0, 0, 255); count_unknown += 1

        text = f"{label} ({conf:.2f})" if (conf is not None and label != "unknown") else label

        cv2.rectangle(frame, (x1 + fx1, y1 + fy1), (x1 + fx2, y1 + fy2), color, 2)
        cv2.putText(frame, text, (x1 + fx1, max(15, y1 + fy1 - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, UI_SCALE, color, UI_THICK)

    now = time.time()
    fps_now = 1.0 / (now - last_time + 1e-6)
    last_time = now

    overlay = frame.copy()
    hud_h = int(30 + 4 * LINE_GAP)
    cv2.rectangle(overlay, (5, 5), (260, hud_h), (0, 0, 0), -1)
    frame = cv2.addWeighted(overlay, 0.4, frame, 0.6, 0)

    total = count_glasses + count_noglasses + count_unknown
    y0 = int(30 * UI_SCALE)

    cv2.putText(frame, f"FPS: {fps_now:.1f}", (10, y0),
                cv2.FONT_HERSHEY_SIMPLEX, UI_SCALE, (255, 255, 255), UI_THICK)
    cv2.putText(frame, f"Total: {total}", (10, y0 + LINE_GAP),
                cv2.FONT_HERSHEY_SIMPLEX, UI_SCALE, (255, 255, 255), UI_THICK)
    cv2.putText(frame, f"Glasses: {count_glasses}", (10, y0 + 2 * LINE_GAP),
                cv2.FONT_HERSHEY_SIMPLEX, UI_SCALE, (0, 255, 0), UI_THICK)
    cv2.putText(frame, f"No Glasses: {count_noglasses}", (10, y0 + 3 * LINE_GAP),
                cv2.FONT_HERSHEY_SIMPLEX, UI_SCALE, (255, 0, 0), UI_THICK)
    cv2.putText(frame, f"Unknown: {count_unknown}", (10, y0 + 4 * LINE_GAP),
                cv2.FONT_HERSHEY_SIMPLEX, UI_SCALE, (0, 0, 255), UI_THICK)

    cv2.imshow("Video Profile-Aware Glasses", frame)
    if writer is not None:
        writer.write(frame)

    key = cv2.waitKey(1) & 0xFF
    if key == 27:
        break

cap.release()
if writer is not None:
    writer.release()
cv2.destroyAllWindows()

print("Done. Output saved to:", OUTPUT_PATH if SAVE_OUTPUT else "(not saved)")



0: 384x640 1 person, 21.5ms
Speed: 29.0ms preprocess, 21.5ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 20.1ms
Speed: 4.0ms preprocess, 20.1ms inference, 2.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 20.0ms
Speed: 2.8ms preprocess, 20.0ms inference, 2.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 20.0ms
Speed: 3.5ms preprocess, 20.0ms inference, 2.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 19.9ms
Speed: 3.9ms preprocess, 19.9ms inference, 3.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 21.1ms
Speed: 3.3ms preprocess, 21.1ms inference, 2.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 20.0ms
Speed: 3.5ms preprocess, 20.0ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 20.1ms
Speed: 6.4ms preprocess, 20.1ms inference, 2.8ms postprocess per image at shape (1, 3, 3

In [1]:
# ==========================================
# Profile-Aware Glasses Detection (Webcam)
# ==========================================

import cv2
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import models
from ultralytics import YOLO
import mediapipe as mp
from PIL import Image
import time

# =====================
# Config
# =====================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GLASSES_MODEL_PATH = r"D:\VISION_2\Final_Glass_Detection\resnet18_glasses.pth"

CAM_INDEX = 0
YOLO_MODEL = "yolov8n.pt"

# Thresholds (Profile-Aware)
FRONTAL_THRESH = 0.65
PROFILE_THRESH = 0.50
ASPECT_PROFILE_CUTOFF = 0.75  

# Optional: Only run CNN every N frames (speed)
CNN_EVERY_N_FRAMES = 1  # 1=every frame, 2=every 2nd frame, ...

# =====================
# Load Models
# =====================
yolo = YOLO(YOLO_MODEL)

mp_face = mp.solutions.face_detection
face_detector = mp_face.FaceDetection(model_selection=1, min_detection_confidence=0.6)

glasses_model = models.resnet18(weights=None)
glasses_model.fc = nn.Linear(glasses_model.fc.in_features, 2)
glasses_model.load_state_dict(torch.load(GLASSES_MODEL_PATH, map_location=DEVICE))
glasses_model = glasses_model.to(DEVICE)
glasses_model.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# =====================
# Quality Gate (junk-only)
# =====================
def quality_gate(face_crop):
    h, w = face_crop.shape[:2]
    if h < 48 or w < 48:
        return False
    gray = cv2.cvtColor(face_crop, cv2.COLOR_BGR2GRAY)
    if gray.mean() < 45:
        return False
    return True

def predict_glasses(face_crop):
    img = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(img)
    img = transform(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        outputs = glasses_model(img)
        probs = torch.softmax(outputs, dim=1)
        conf, pred = torch.max(probs, 1)
    label = "glasses" if pred.item() == 0 else "no_glasses"
    return label, float(conf.item())

# =====================
# Webcam
# =====================
cap = cv2.VideoCapture(CAM_INDEX)
if not cap.isOpened():
    raise RuntimeError("Cannot open webcam")

frame_id = 0
last_time = time.time()

cv2.namedWindow("Webcam Profile-Aware Glasses", cv2.WINDOW_NORMAL)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_id += 1
    H, W = frame.shape[:2]

    # UI scale
    UI_SCALE = max(0.35, min(W / 800, 0.7))
    UI_THICK = 1 if W < 600 else 2
    LINE_GAP = int(22 * UI_SCALE)

    # Counters
    count_glasses = 0
    count_noglasses = 0
    count_unknown = 0

    # YOLO person detection (BGR is OK)
    results = yolo(frame)

    for box in results[0].boxes:
        cls = int(box.cls[0])
        if cls != 0:
            continue

        x1, y1, x2, y2 = map(int, box.xyxy[0])
        person_crop = frame[y1:y2, x1:x2]
        if person_crop.size == 0:
            continue

        rgb_person = cv2.cvtColor(person_crop, cv2.COLOR_BGR2RGB)
        faces = face_detector.process(rgb_person)

        if not faces.detections:
            continue

        # take first face per person
        det = faces.detections[0]
        bbox = det.location_data.relative_bounding_box
        ph, pw, _ = person_crop.shape

        fx1 = int(bbox.xmin * pw)
        fy1 = int(bbox.ymin * ph)
        fx2 = int((bbox.xmin + bbox.width) * pw)
        fy2 = int((bbox.ymin + bbox.height) * ph)

        # clamp + optional padding
        pad = 8
        fx1 = max(0, fx1 - pad)
        fy1 = max(0, fy1 - pad)
        fx2 = min(pw, fx2 + pad)
        fy2 = min(ph, fy2 + pad)

        face_crop = person_crop[fy1:fy2, fx1:fx2]
        if face_crop.size == 0:
            continue

        conf = None
        label = "unknown"

        if quality_gate(face_crop):
            # optional speed: run CNN every N frames
            if frame_id % CNN_EVERY_N_FRAMES == 0:
                pred_label, pred_conf = predict_glasses(face_crop)
                conf = pred_conf

                # profile-aware threshold
                aspect = (fx2 - fx1) / (fy2 - fy1 + 1e-6)
                thresh = PROFILE_THRESH if aspect < ASPECT_PROFILE_CUTOFF else FRONTAL_THRESH

                if pred_conf >= thresh:
                    label = pred_label
                else:
                    label = "unknown"
            else:
                # if skipping CNN this frame, keep unknown (simple)
                label = "unknown"

        # color + counts
        if label == "glasses":
            color = (0, 255, 0); count_glasses += 1
        elif label == "no_glasses":
            color = (255, 0, 0); count_noglasses += 1
        else:
            color = (0, 0, 255); count_unknown += 1

        text = f"{label} ({conf:.2f})" if (conf is not None and label != "unknown") else label

        # draw face box (not whole person)
        cv2.rectangle(frame, (x1 + fx1, y1 + fy1), (x1 + fx2, y1 + fy2), color, 2)
        cv2.putText(frame, text, (x1 + fx1, max(15, y1 + fy1 - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, UI_SCALE, color, UI_THICK)

    # FPS
    now = time.time()
    fps = 1.0 / (now - last_time + 1e-6)
    last_time = now

    # HUD background
    overlay = frame.copy()
    hud_h = int(30 + 4 * LINE_GAP)
    cv2.rectangle(overlay, (5, 5), (260, hud_h), (0, 0, 0), -1)
    frame = cv2.addWeighted(overlay, 0.4, frame, 0.6, 0)

    total = count_glasses + count_noglasses + count_unknown
    y0 = int(30 * UI_SCALE)

    cv2.putText(frame, f"FPS: {fps:.1f}", (10, y0),
                cv2.FONT_HERSHEY_SIMPLEX, UI_SCALE, (255, 255, 255), UI_THICK)
    cv2.putText(frame, f"Total: {total}", (10, y0 + LINE_GAP),
                cv2.FONT_HERSHEY_SIMPLEX, UI_SCALE, (255, 255, 255), UI_THICK)
    cv2.putText(frame, f"Glasses: {count_glasses}", (10, y0 + 2 * LINE_GAP),
                cv2.FONT_HERSHEY_SIMPLEX, UI_SCALE, (0, 255, 0), UI_THICK)
    cv2.putText(frame, f"No Glasses: {count_noglasses}", (10, y0 + 3 * LINE_GAP),
                cv2.FONT_HERSHEY_SIMPLEX, UI_SCALE, (255, 0, 0), UI_THICK)
    cv2.putText(frame, f"Unknown: {count_unknown}", (10, y0 + 4 * LINE_GAP),
                cv2.FONT_HERSHEY_SIMPLEX, UI_SCALE, (0, 0, 255), UI_THICK)

    cv2.imshow("Webcam Profile-Aware Glasses", frame)

    key = cv2.waitKey(1) & 0xFF
    if key == 27:  # ESC
        break

cap.release()
cv2.destroyAllWindows()



0: 480x640 1 person, 1 chair, 2 potted plants, 61.9ms
Speed: 5.4ms preprocess, 61.9ms inference, 20.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 chair, 2 potted plants, 20.0ms
Speed: 1.8ms preprocess, 20.0ms inference, 3.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 chair, 2 potted plants, 10.8ms
Speed: 1.5ms preprocess, 10.8ms inference, 1.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 chair, 2 potted plants, 10.7ms
Speed: 1.1ms preprocess, 10.7ms inference, 1.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 chair, 2 potted plants, 9.3ms
Speed: 1.4ms preprocess, 9.3ms inference, 1.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 chair, 1 potted plant, 9.5ms
Speed: 0.9ms preprocess, 9.5ms inference, 1.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 chair, 1 potted plant, 9.6ms
Speed: 1.0ms preprocess, 9.6ms inferenc